# ERK-KTR Full FOV Stimulation Pipeline (RTMSequence)

This notebook runs a multi-phase ERK-KTR optogenetic experiment with full-FOV light stimulation on the Jungfrau microscope.

In [1]:
import os
import time
import pandas as pd
from faro.core.data_structures import (
    Channel,  # basic imaging channel (config, exposure, group)
    PowerChannel,  # imaging channel with light-source power control (adds power 0-100)
    RTMSequence,  # defines one phase of the acquisition (time plan, channels, stim, etc.)
    SegmentationMethod,
    combine,  # compose multi-phase experiments along an axis (t or p)
)

from faro.agents import (
    FOVFinderAgent,
    ComposedAgent,
    FOVCondition,
    FOVConditionMonitorAgent,
    WellPattern,  # pairs a well with a build_sequence(fovs) -> RTMSequence callback
    resolve_well_patterns,  # find ALL FOVs up front, combine + batch into one run
    run_well_patterns,  # find + run in sequential batches (6 wells at a time)
)


import faro.core.utils as utils

In [2]:
patterns_to_test = pd.read_csv("mixed_normal_patterns.csv")
patterns_to_test["value"] = patterns_to_test["value"].round(0).astype(int)
patterns_to_test["uid"] = patterns_to_test["uid"].astype(int)
patterns_to_test["time"] = patterns_to_test["time"].astype(int)

### Experimental Settings

In [3]:
from faro.microscope.pertzlab.jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup(
    "TTL_ERK"
)  # select the channel group configured in Micro-Manager

In [4]:
WELLS = []
START_COL = 2
END_COL = 12
for i, row in enumerate("ABCDEFGH"):
    cols = (
        range(START_COL, END_COL + 1)
        if i % 2 == 0
        else range(END_COL, START_COL - 1, -1)
    )
    WELLS.extend(f"{row}{c}" for c in cols)

In [5]:
## Configuration
SLEEP_BEFORE_EXPERIMENT_START_in_H = (
    8  # delay before acquisition (hours); 0 = start immediately
)

## Storage -- all output (zarr, tracks, TIFFs) goes under this directory
base_path = "E:\\Alex"
experiment_name = "2026-06-23_FreePatternStim_Jungfrau_v1"
path = os.path.join(base_path, experiment_name)

## Stimulation channel -- light used for optogenetic activation
stim_channel = PowerChannel(
    config="CyanStim",  # Micro-Manager channel preset name
    exposure=100,  # stimulation pulse duration (ms)
    group="TTL_ERK",  # Micro-Manager channel group
    power=10,  # light source intensity (0-100)
)

## Imaging channels -- acquired at every timepoint; order matters (channel 0 is used for segmentation)
imaging_channels = (
    PowerChannel(
        config="miRFP", exposure=150, group="TTL_ERK", power=95
    ),  # nuclear marker
    PowerChannel(
        config="mScarlet3", exposure=150, group="TTL_ERK", power=95
    ),  # ERK-KTR reporter
)

## Optocheck channel -- reference channel to verify optogenetic tool expression (longer exposure)
optocheck_channel = PowerChannel(
    config="mCitrine", exposure=600, group="TTL_ERK", power=95
)

### Pipeline Setup

In [6]:
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.optocheck import OptoCheckFE
from faro.segmentation.cellpose_v4 import CellposeV4

segmentators = [
    SegmentationMethod(
        name="labels",  # label layer name in the zarr store
        segmentation_class=CellposeV4(
            custom_model_path=r"E:\models\cellpose\LifeActH2B_mixed_with_only_H2B_v1",
            min_size=100,
        ),
        save_tracked=True,  # persist tracked label masks alongside raw segmentation
    )
]

stimulator = StimWholeFOV()  # illuminate entire FOV (no DMD patterning)
feature_extractor = FE_ErkKtr("labels")  # cytoplasmic/nuclear ratio using "labels" mask
tracker = TrackerTrackpy(search_range=50)  # max 50 px displacement between frames
optocheck = OptoCheckFE(used_mask="labels")  # measure optogenetic reporter per cell

from faro.core.pipeline import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_ref=optocheck,  # runs only on ref_frames (optocheck timepoints)
)

from faro.core.controller import Controller
from faro.core.writers import OmeZarrWriter

writer = OmeZarrWriter(storage_path=path)  # saves images + stim readout into OME-Zarr

Directory E:\Alex\2026-06-23_FreePatternStim_Jungfrau_v1\tracks already exists


In [7]:
# =============================================================================
# Stimulation patterns  +  per-well FOV-finder configuration
# =============================================================================
PLATE_CALIBRATION_PATH = (
    r".\calib_plate_96.json"  # <-- WellPlatePlan JSON from the MDA plate widget
)

# FOV-finder settings (tune for your plate / cell density) --------------------
FOVS_PER_WELL = 3  # FOVs kept per well
N_CANDIDATES_PER_WELL = 15  # candidates scanned per well (> FOVS_PER_WELL)
FOV_BORDER_UM = 1300  # keep candidates this far from the well edge
FOV_MIN_DISTANCE_UM = 700  # min spacing between candidates in a well
FOV_MIN_CELLS = 20  # reject FOVs with fewer cells
FOV_MAX_CELLS = 150  # reject overly-confluent FOVs
TIME_PER_FOV = 3.3  # seconds to image one FOV (used for batching)
N_PARALLEL_FOVS = 18  # FOVs imaged per batch (e.g. 6 wells x 3 FOVs)
TIME_BETWEEN_TIMESTEPS = 60.0  # seconds between timepoints

# Accept a well's FOVs only where the ERK-KTR signal is usable
# (CNR below 1.0 in >= 70 % of cells).
ready = FOVCondition("cnr", "below", 1.0, min_fraction=0.7)


# --- Map one CSV waveform (uid) -> a per-well RTMSequence pattern ------------
# A frame is stimulated where the waveform value > threshold, AND that frame's
# `value` sets its stim pulse duration (stim_exposure, in ms). The FOVs are
# filled in at runtime, so build_sequence() takes them as input.
def stim_pattern_from_waveform(
    well, times, values, *, threshold=0.0, rtm_metadata=None
):
    """Build a WellPattern from a per-frame stim waveform.

    A frame ``t`` is a stim frame iff ``value[t] > threshold``, and that frame's
    pulse duration is ``value[t]`` ms (``stim_exposure``). RTMSequence maps
    ``stim_exposure[i]`` to the i-th frame of ``sorted(stim_frames)``, so both
    are built in sorted order.
    """
    val_by_t = {int(t): float(v) for t, v in zip(times, values)}
    stim_frames_sorted = sorted(t for t, v in val_by_t.items() if v > threshold)
    stim_frames = frozenset(stim_frames_sorted)
    # Per-frame pulse duration (ms), aligned to sorted(stim_frames).
    stim_exposure = [val_by_t[t] for t in stim_frames_sorted] or None
    n_frames = int(max(times)) + 1
    base_meta = dict(rtm_metadata or {})

    def build(fovs):  # fovs: list[FovPosition] located at runtime
        return RTMSequence(
            time_plan={"interval": TIME_BETWEEN_TIMESTEPS, "loops": n_frames},
            stage_positions=fovs,
            channels=imaging_channels,
            stim_channels=(stim_channel,),
            stim_frames=stim_frames,
            stim_exposure=stim_exposure,  # value -> per-frame pulse duration (ms)
            ref_channels=(optocheck_channel,),
            ref_frames=frozenset({n_frames - 1}),  # optocheck on the last frame
            rtm_metadata={"well": well, **base_meta},
        )

    return WellPattern(well=well, build_sequence=build)


# One pattern per uid -> one well (in WELLS order) ---------------------------
uids = sorted(patterns_to_test["uid"].unique())
pattern_wells = WELLS[: len(uids)]
assert len(WELLS) >= len(
    uids
), f"Need {len(uids)} wells for {len(uids)} patterns; only {len(WELLS)} defined."

patterns = []
for uid, well in zip(uids, pattern_wells):
    g = patterns_to_test[patterns_to_test.uid == uid].sort_values("time")
    patterns.append(
        stim_pattern_from_waveform(
            well=well,
            times=g["time"].to_numpy(),
            values=g["value"].to_numpy(),  # value > 0 -> stim frame; value = pulse ms
            rtm_metadata={
                "uid": int(uid),
                "treatment_name": f"pattern_{uid}",
            },
        )
    )

# Shared FOV-finder config (the resolver builds one finder per well from this).
finder_kwargs = dict(
    well_plate_plan=PLATE_CALIBRATION_PATH,
    fovs_per_well=FOVS_PER_WELL,
    n_candidates_per_well=N_CANDIDATES_PER_WELL,
    border_um=FOV_BORDER_UM,
    min_distance_um=FOV_MIN_DISTANCE_UM,
    min_cells=FOV_MIN_CELLS,
    max_cells=FOV_MAX_CELLS,
    imaging_channels=imaging_channels,
    segmentator=segmentators[0].segmentation_class,  # reuse the loaded Cellpose model
    seg_channel_index=0,
    feature_extractor=FE_ErkKtr("labels"),
    fov_conditions=[ready],
    selection_mode="extremes",
    z=None,
)

print(f"{len(patterns)} patterns queued -> wells {pattern_wells}")

88 patterns queued -> wells ['A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'B12', 'B11', 'B10', 'B9', 'B8', 'B7', 'B6', 'B5', 'B4', 'B3', 'B2', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'D12', 'D11', 'D10', 'D9', 'D8', 'D7', 'D6', 'D5', 'D4', 'D3', 'D2', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'F12', 'F11', 'F10', 'F9', 'F8', 'F7', 'F6', 'F5', 'F4', 'F3', 'F2', 'G2', 'G3', 'G4', 'G5', 'G6', 'G7', 'G8', 'G9', 'G10', 'G11', 'G12', 'H12', 'H11', 'H10', 'H9', 'H8', 'H7', 'H6', 'H5', 'H4', 'H3', 'H2']


In [8]:
patterns[0].build_sequence([])  # test build_sequence() with empty FOVs

RTMSequence(channels=(Channel(config='miRFP', group='TTL_ERK', exposure=150.0), Channel(config='mScarlet3', group='TTL_ERK', exposure=150.0)), time_plan=TIntervalLoops(interval=datetime.timedelta(seconds=60), loops=91), stim_channels=(PowerChannel(config='CyanStim', exposure=100, group='TTL_ERK', power=10),), stim_frames=frozenset({10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68}), stim_exposure=[53.0, 63.0, 74.0, 86.0, 98.0, 110.0, 123.0, 134.0, 146.0, 156.0, 164.0, 172.0, 177.0, 181.0, 183.0, 183.0, 182.0, 178.0, 174.0, 167.0, 160.0, 152.0, 143.0, 134.0, 124.0, 115.0, 105.0, 96.0, 87.0, 79.0], ref_channels=(PowerChannel(config='mCitrine', exposure=600, group='TTL_ERK', power=95),), ref_frames=frozenset({90}), rtm_metadata={'well': 'A2', 'uid': 0, 'treatment_name': 'pattern_0'})

### GUI (optional)

Opens a napari viewer with the Micro-Manager widget for live camera view and manual focusing. FOV positions are **not** picked here — they are located automatically per well by the FOV finder during the run.

In [9]:
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(
    viewer, mmcore=mic.mmc
)  # widget to control Micro-Manager from napari
viewer.window.add_dock_widget(mm_wdg)  # dock Micro-Manager controls in napari

### Run — find FOVs and stimulate, one batch of wells at a time (mode B)

`run_well_patterns` plugs the FOV finder into the experiment **during** the run:

1. Scan `WELLS_PER_BATCH` wells (→ `WELLS_PER_BATCH × FOVS_PER_WELL` FOVs).
2. Build those FOVs' RTM events **on the fly** and run that batch's time-lapse to completion.
3. Scan + run the next batch, and so on.

Because each batch's events are generated *after* its scan finishes (and run via `continue_experiment`, which restarts the per-batch time-lapse clock from 0), **the finder's scan time never corrupts the time-lapse schedule**. Every FOV is a physically distinct position, so all batches accumulate into a single store / one tracks DataFrame — no per-phase bookkeeping.

It's launched via `ctrl.run_orchestrator_async(...)`, which runs the (blocking) batch loop on a controller worker thread and returns an **`OrchestratorHandle`** immediately, so napari stays responsive. The handle gives two levels of progress through one object:

- **orchestrator level** — `run_handle.status()` → `state`, `step`/`n_steps`, `message` (e.g. `"batch 3/15: C2, C3, …"`);
- **drill-down** — `run_handle.current_run` is the `RunHandle` of the batch currently acquiring (`status().n_events_acquired / n_events_total`, `current_fov`); `currentRunChanged` fires when it advances to the next batch.

`run_handle.cancel()` aborts the in-flight batch and halts between batches; `run_handle.wait()` blocks until done. This is the **same async/cancel/drill-down contract every multi-run agent will use** — e.g. `ctrl.run_orchestrator_async(composed_agent.run)` for a BO experiment.

> **Mode A alternative** (all FOVs found up front, one combined acquisition): `events = resolve_well_patterns(mic, patterns, finder_kwargs=finder_kwargs, time_per_fov=TIME_PER_FOV, n_parallel=N_PARALLEL_FOVS).events` then `ctrl.run_experiment(events)`. Use this only when the up-front scan of every well is acceptable.

In [10]:
# Pre-flight: validate a representative pattern's events before committing to the run.
_probe_ctrl = Controller(mic, pipeline, writer=None)  # no writer; validation only
from faro.core.utils import FovPosition

_probe_fovs = [FovPosition(x=0.0, y=0.0, z=None, name="A2_0000")]
_probe_events = list(patterns[0].build_sequence(_probe_fovs))
ok = _probe_ctrl.validate_events(_probe_events)
print("pattern[0] events valid:", ok, "| n events:", len(_probe_events))
# Also eyeball the per-frame stim exposures that will actually fire:
print(
    "stim exposures (ms):",
    sorted({c.exposure for e in _probe_events for c in e.stim_channels}),
)

pattern[0] events valid: True | n events: 91
stim exposures (ms): [53.0, 63.0, 74.0, 79.0, 86.0, 87.0, 96.0, 98.0, 105.0, 110.0, 115.0, 123.0, 124.0, 134.0, 143.0, 146.0, 152.0, 156.0, 160.0, 164.0, 167.0, 172.0, 174.0, 177.0, 178.0, 181.0, 182.0, 183.0]


In [11]:
WELLS_PER_BATCH = 6  # wells found + run together (6 x 3 FOVs = 18 FOVs per batch)

ctrl = Controller(mic, pipeline, writer=writer)

# Live status + pause/stop buttons (re-binds when each batch's run starts).
from faro.widgets import ExperimentStatusWidget

viewer.window.add_dock_widget(
    ExperimentStatusWidget(ctrl), name="experiment status", area="right"
)

# Optional delay before the first batch (set SLEEP_BEFORE_EXPERIMENT_START_in_H above).
for _ in range(int(SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600)):
    time.sleep(1)

# Launch the batch-sequential run on a controller worker thread; returns at once
# so napari stays responsive. run_orchestrator_async injects the OrchestratorHandle
# as `progress`, so run_well_patterns reports "batch i/n" and is cancellable.
#   run_handle.status()           -> orchestrator level (state, step/n_steps, message)
#   run_handle.current_run        -> the RunHandle of the batch currently acquiring
#   run_handle.cancel()           -> abort the in-flight batch + stop between batches
#   run_handle.wait()             -> block until all batches finish
run_handle = ctrl.run_orchestrator_async(
    run_well_patterns,
    ctrl,
    mic,
    patterns,
    wells_per_batch=WELLS_PER_BATCH,
    finder_kwargs=finder_kwargs,
    time_per_fov=TIME_PER_FOV,
    n_parallel=N_PARALLEL_FOVS,
    stim_mode="current",
    finish=True,  # close the store after the last batch
)
print("Experiment launched. run_handle.status().state ->", run_handle.status().state)


=== run_well_patterns: batch 1/15 (6 wells: ['A2', 'A3', 'A4', 'A5', 'A6', 'A7']) ===Experiment launched. run_handle.status().state -> running

[resolve_well_patterns] 1/6: finding FOVs in well 'A2' ...


C:\Users\Jungfrau\Documents\alandolt\code\faro.worktrees\feat-agent_mode\faro\agents\fov_finder.py:601: UserWarning: Unable to generate 15 non-overlapping points. Only 14 points were found.
  return np.array([(p.x, p.y) for p in rp], dtype=float)


[FOVFinderAgent] Phase 0 — selected FOVs:
    A2_0000: 16 cells (rejected: below_min_cells)
    A2_0001: 10 cells (rejected: below_min_cells)
    A2_0002: 7 cells (rejected: below_min_cells)
[resolve_well_patterns]   well 'A2': 3 FOV(s) -> ['A2_0000', 'A2_0001', 'A2_0002']
[resolve_well_patterns] 2/6: finding FOVs in well 'A3' ...
[FOVFinderAgent] Phase 0 — selected FOVs:
    A3_0000: 55 cells  cnr(mean=1.233, 21% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
    A3_0001: 22 cells  cnr(mean=1.014, 50% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
    A3_0002: 18 cells (rejected: below_min_cells)
[resolve_well_patterns]   well 'A3': 3 FOV(s) -> ['A3_0000', 'A3_0001', 'A3_0002']
[resolve_well_patterns] 3/6: finding FOVs in well 'A4' ...
[FOVFinderAgent] Phase 0 — selected FOVs:
    A4_0000: 85 cells  cnr(mean=0.887, 65% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
    A4_0001: 35 cells  cnr(mean=0.937, 53% below 1) (rejected: condition_failed:cnr_below_1

C:\Users\Jungfrau\Documents\alandolt\code\faro.worktrees\feat-agent_mode\faro\agents\fov_finder.py:601: UserWarning: Unable to generate 15 non-overlapping points. Only 14 points were found.
  return np.array([(p.x, p.y) for p in rp], dtype=float)


[FOVFinderAgent] Phase 0 — selected FOVs:
    A6_0000: 48 cells  cnr(mean=1.059, 40% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
    A6_0001: 26 cells  cnr(mean=1.060, 42% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
    A6_0002: 20 cells  cnr(mean=0.964, 59% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
[resolve_well_patterns]   well 'A6': 3 FOV(s) -> ['A6_0000', 'A6_0001', 'A6_0002']
[resolve_well_patterns] 6/6: finding FOVs in well 'A7' ...
[FOVFinderAgent] Phase 0 — selected FOVs:
    A7_0000: 125 cells  cnr(mean=0.927, 62% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
    A7_0001: 21 cells  cnr(mean=0.992, 42% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
    A7_0002: 19 cells (rejected: below_min_cells)
[resolve_well_patterns]   well 'A7': 3 FOV(s) -> ['A7_0000', 'A7_0001', 'A7_0002']
[resolve_well_patterns] resolved 6 pattern(s), 18 FOV(s) total, 1638 event(s).
[run_well_patterns] batch 0: 18 FOVs (p 0..17); run_experimen

...

[FOVFinderAgent] Phase 0 — selected FOVs:
    G9_0000: 31 cells  cnr(mean=0.929, 60% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
    G9_0001: 23 cells  cnr(mean=1.001, 44% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
    G9_0002: 19 cells (rejected: below_min_cells)
[resolve_well_patterns]   well 'G9': 3 FOV(s) -> ['G9_0000', 'G9_0001', 'G9_0002']
[resolve_well_patterns] 3/6: finding FOVs in well 'G10' ...
[FOVFinderAgent] Phase 0 — selected FOVs:
    G10_0000: 20 cells  cnr(mean=0.849, 75% below 1)
    G10_0001: 21 cells  cnr(mean=1.003, 47% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
    G10_0002: 18 cells (rejected: below_min_cells)
[resolve_well_patterns]   well 'G10': 3 FOV(s) -> ['G10_0000', 'G10_0001', 'G10_0002']
[resolve_well_patterns] 4/6: finding FOVs in well 'G11' ...
[FOVFinderAgent] Phase 0 — selected FOVs:
    G11_0000: 21 cells  cnr(mean=0.960, 61% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
    G11_0001: 18 cells (reje

C:\Users\Jungfrau\Documents\alandolt\code\faro.worktrees\feat-agent_mode\faro\agents\fov_finder.py:601: UserWarning: Unable to generate 15 non-overlapping points. Only 14 points were found.
  return np.array([(p.x, p.y) for p in rp], dtype=float)


[FOVFinderAgent] Phase 0 — selected FOVs:
    H12_0000: 89 cells  cnr(mean=0.796, 81% below 1)
    H12_0001: 107 cells  cnr(mean=0.832, 78% below 1)
    H12_0002: 114 cells  cnr(mean=0.867, 63% below 1) (rejected: condition_failed:cnr_below_1.0@0.70)
[resolve_well_patterns]   well 'H12': 3 FOV(s) -> ['H12_0000', 'H12_0001', 'H12_0002']
[resolve_well_patterns] resolved 6 pattern(s), 18 FOV(s) total, 1638 event(s).
[run_well_patterns] batch 12: 18 FOVs (p 216..233); continue_experiment ...

=== run_well_patterns: batch 14/15 (6 wells: ['H11', 'H10', 'H9', 'H8', 'H7', 'H6']) ===
[resolve_well_patterns] 1/6: finding FOVs in well 'H11' ...


C:\Users\Jungfrau\Documents\alandolt\code\faro.worktrees\feat-agent_mode\faro\agents\fov_finder.py:601: UserWarning: Unable to generate 15 non-overlapping points. Only 14 points were found.
  return np.array([(p.x, p.y) for p in rp], dtype=float)


[FOVFinderAgent] Phase 0 — selected FOVs:
    H11_0000: 34 cells  cnr(mean=0.780, 82% below 1)
    H11_0001: 13 cells (rejected: below_min_cells)
    H11_0002: 11 cells (rejected: below_min_cells)
[resolve_well_patterns]   well 'H11': 3 FOV(s) -> ['H11_0000', 'H11_0001', 'H11_0002']
[resolve_well_patterns] 2/6: finding FOVs in well 'H10' ...
[FOVFinderAgent] Phase 0 — selected FOVs:
    H10_0000: 17 cells (rejected: below_min_cells)
    H10_0001: 13 cells (rejected: below_min_cells)
    H10_0002: 12 cells (rejected: below_min_cells)
[resolve_well_patterns]   well 'H10': 3 FOV(s) -> ['H10_0000', 'H10_0001', 'H10_0002']
[resolve_well_patterns] 3/6: finding FOVs in well 'H9' ...
[FOVFinderAgent] Phase 0 — selected FOVs:
    H9_0000: 26 cells  cnr(mean=0.863, 79% below 1)
    H9_0001: 59 cells  cnr(mean=0.781, 84% below 1)
    H9_0002: 89 cells  cnr(mean=0.732, 87% below 1)
[resolve_well_patterns]   well 'H9': 3 FOV(s) -> ['H9_0000', 'H9_0001', 'H9_0002']
[resolve_well_patterns] 4/6: findi

In [11]:
run_handle.cancel()  # cancel the run (if needed) -- stops the current batch + prevents future batches

### Monitor / cancel (optional)

Re-run the cell below any time to snapshot progress without blocking. Cancel the whole run with `run_handle.cancel()`.

In [ ]:
# Snapshot progress without blocking (re-run anytime while the experiment runs).
# The ExperimentStatusWidget already renders the current batch's per-event detail;
# this prints the orchestrator-level view + a drill-down into the live batch.
s = run_handle.status()
print(f"orchestrator: {s.state}" + (f"  |  {s.message}" if s.message else ""))

cur = run_handle.current_run
if cur is not None:
    cs = cur.status()
    print(
        f"  current batch: {cs.state}  "
        f"{cs.n_events_acquired}/{cs.n_events_total} events"
        + (f", FOV {cs.current_fov}" if cs.current_fov is not None else "")
    )

# Stop early (aborts the in-flight batch, then halts between batches):
#   run_handle.cancel()

### Post-processing

`run_handle.wait()` blocks until every batch has finished (`run_well_patterns(finish=True)`
already flushed the pipeline and closed the zarr store). Then we merge the per-FOV
track parquet files into a single `exp_data.parquet`.

In [12]:
run_handle.wait()  # block until all batches finish (run already closed the store)

utils.generate_exp_data_from_tracks(path)  # merge per-FOV tracks into exp_data.parquet

In [14]:
pd.read_parquet(os.path.join(path, "exp_data.parquet"))

,label,x,y,well,uid,treatment_name,fov,timestep,fname,time,...,mean_intensity_C0_ring,mean_intensity_C1_ring,median_intensity_C0_ring,median_intensity_C1_ring,cnr_mean,cnr,stim_power,stim_exposure,ref_mean_intensity,time_offset
0,1,24.079023,762.409483,A2,0,pattern_0,0,0,000_00000,0.000000,...,266.244240,1755.576037,262.0,1671.0,1.004681,0.969539,NaN,NaN,NaN,NaN
1,2,115.262712,664.752119,A2,0,pattern_0,0,0,000_00000,0.000000,...,294.264865,1611.097297,289.0,1627.0,0.783638,0.791535,NaN,NaN,NaN,NaN
2,3,232.250340,644.259864,A2,0,pattern_0,0,0,000_00000,0.000000,...,305.343750,2328.839286,303.5,2352.5,0.956920,0.984310,NaN,NaN,NaN,NaN
3,4,297.828194,186.046256,A2,0,pattern_0,0,0,000_00000,0.000000,...,295.718232,2777.309392,294.0,2669.0,0.918091,0.889963,NaN,NaN,NaN,NaN
4,5,315.103960,34.371287,A2,0,pattern_0,0,0,000_00000,0.000000,...,272.787356,1332.327586,273.5,1310.0,0.946463,0.938395,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9018,98,845.651226,255.059946,A5,3,pattern_3,9,90,009_00090,5429.700195,...,277.022727,7636.645455,275.0,7769.5,1.088046,1.114146,NaN,NaN,2299.754768,NaN
9019,99,873.545939,309.849534,A5,3,pattern_3,9,90,009_00090,5429.700195,...,280.504274,2468.282051,280.0,2389.5,0.735560,0.718863,NaN,NaN,3669.704394,NaN
9020,100,907.555556,202.073552,A5,3,pattern_3,9,90,009_00090,5429.700195,...,259.535211,1744.380282,259.0,1734.0,0.768018,0.769299,NaN,NaN,1247.978091,NaN
9021,101,951.501471,448.010294,A5,3,pattern_3,9,90,009_00090,5429.700195,...,267.648148,1346.162037,268.5,1309.5,0.895484,0.866071,NaN,NaN,1406.601471,NaN
